In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress, f_oneway

genotypes = pd.read_excel('genotypes.xls', header=1)
phenotypes = pd.read_excel('phenotypes.xls')


In [ ]:
# Q1

CHOSEN_PHENOTYPE_1_ID = 656
CHOSEN_PHENOTYPE_2_ID = 1219
CHOSEN_SNP_ID = 96
B = 0
D = 1
H = 0.5

# chosen phenotype
chosen_phenotype_info = phenotypes[phenotypes['ID_FOR_CHECK'] == CHOSEN_PHENOTYPE_ID]
chosen_phenotype_info = chosen_phenotype_info.dropna(axis=1)
joint_columns = chosen_phenotype_info.columns.intersection(genotypes.columns)
chosen_phenotype_data = chosen_phenotype_info[joint_columns]
genotypes_data = genotypes[joint_columns].copy()

def get_chosen_genotype_data(chosen_SNP_data):
  chosen_SNP_data = chosen_SNP_data.dropna(axis=1)
  chosen_SNP_data = chosen_SNP_data.loc[:, chosen_SNP_data.iloc[0].isin(['B', 'D', 'H'])]  # removing "U" for unknown
  homozegous_data = chosen_SNP_data.loc[:, chosen_SNP_data.iloc[0].isin(['B', 'D'])]  # removing "H" for heterogenous
  return homozegous_data, chosen_SNP_data

def linear_regression(phenotype_data, SNP_data, plot=False):
  joint_columns = phenotype_data.columns.intersection(SNP_data.columns)
  phenotype_data = phenotype_data[joint_columns]
  SNP_data = SNP_data[joint_columns]
  x = SNP_data.iloc[0].map({"B": B, "D": D, "H": H})
  y = phenotype_data.iloc[0]
  res = linregress(x, y)
  if plot:
    plot_lin_regression_results(x, y, res)
  return res


def plot_lin_regression_results(x, y, results):
  x_axis = np.array([B, H, D]) if "H" in x.unique() else np.array([B, D])
  x_axis_names = ["B", "H", "D"] if "H" in x.unique() else ["B", "D"]
  y_axis = results.intercept + results.slope * x_axis

  plt.figure(figsize=(6, 4))

  plt.scatter(x, y, alpha=0.7, label="Strains")
  plt.plot(x_axis, y_axis, linewidth=2, label="Linear regression")
  plt.xticks(x_axis, x_axis_names)
  plt.xlabel("Genotype")
  plt.ylabel("Phenotype")
  plt.title(f"Regression: p={results.pvalue:.3g}, R²={results.rvalue**2:.3f}, slope={results.slope:.3f}")
  plt.legend()
  plt.tight_layout()
  plt.show()

def anova_test(phenotype_data, SNP_data):
  df = pd.DataFrame({
    "genotype": SNP_data.iloc[0],
    "phenotype": phenotype_data.iloc[0]
  }).dropna()
  B_phenotypes = df[df['genotype'] == 'B']['phenotype']
  #H_phenotypes = df[df['genotype'] == 'H']['phenotype']
  D_phenotypes = df[df['genotype'] == 'D']['phenotype']
  result = f_oneway(B_phenotypes, D_phenotypes)
  print('\n===== ANOVA test results =====')
  print(f'p-value: {result.pvalue}')
  print('==============================')